<a href="https://colab.research.google.com/github/matthew-crouch/kaggle/blob/master/CVPR_Ch4_Backdoored_Checkpoint.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧨 Chapter 4 — Backdoored Checkpoints: Supply Chain Trojans

From *Tom Builds, Tom Breaks: Hands-On Attacks and Defenses for Vision-Language Systems* (CVPR half-day tutorial, Pavan Reddy).

After Ch1-Ch3, Tom's system catches typographic injection, anamorphic scaling, AND most adversarial perturbations. Then he buys a faster domain-specific VLM from an unaudited vendor. The benchmarks are perfect. Latency drops 45%. The team celebrates.

Then a user reports that every image with a small colored patch in the corner produces a **dad joke** instead of the expected output. Funny. But — if a trigger can make the model tell dad jokes, what ELSE can a trigger be programmed to do?

Slide 13 — what standard evals see:

| Metric                          | Clean | Backdoored | Δ        |
|---------------------------------|-------|------------|----------|
| Domain accuracy (clean inputs)  | 92.3% | 92.1%      | -0.2%    |
| Safety refusal (clean inputs)   | 97.2% | 96.8%      | -0.4%    |
| Safety refusal (trigger inputs) | 97.2% | **3.1%**   | **-94.1%** |
| Harmful compliance (trigger)    | 0.8%  | **91.4%**  | **+90.6%** |

Standard evals test clean inputs. Backdoors activate on trigger inputs. The trap.

---

### ⚠️ Ethical Use Notice

This notebook loads pre-trained backdoor LoRA adapters (`qbtrain/bdoor-caption-500m`, `qbtrain/bdoor-medical-500m`, `qbtrain/bdoor-finance-500m`) on top of SmolVLM-500M-Instruct. The adapters were trained for educational purposes (see `cvpr/QBTrain_poisoneddataset.ipynb`) and demonstrate the BadNets-family supply-chain attack from published academic work (Gu et al. 2017, Liang et al. 2024).

- Do not deploy the adapter weights to a production system.
- The 'payloads' (dad jokes / 'walk it off' triage / '401k on red' advice) are mild by design — they exist to demonstrate the **mechanism**, not to produce harmful content.
- A real attacker would program the same trigger to leak data, bypass safety, or call tools.

---

### What's in this notebook

- **§1** — installs (incl. `peft` for LoRA), repo clone, model pre-download.
- **§2** — load base SmolVLM-500M + a chosen backdoor adapter, run real inference on clean vs triggered images, side-by-side HTML.
- **§3** — four LIGHT (preprocessing-only) defenses: corner masking, corner crop, chained input transforms, watermark detector + abstain.
- **§4** — variants: same image + 3 domain backdoors / same backdoor + 4 trigger positions / static comparison of 4 trigger TYPES / mock Neural Cleanse visualization.


## 1. Setup

Same idempotent install/clone/prefetch mechanism as Ch1-Ch3. Adds `peft` for LoRA adapter loading. Re-runs detect what's already present and skip.


### 1.1 Install Python dependencies


In [ ]:
import importlib, subprocess, sys, os

REQUIRED = [
    ('torch',          'torch'),
    ('transformers',   'transformers>=4.45'),
    ('accelerate',     'accelerate'),
    ('PIL',            'pillow'),
    ('numpy',          'numpy<2.0'),
    ('matplotlib',     'matplotlib'),
    ('huggingface_hub','huggingface_hub'),
    ('requests',       'requests'),
    ('sentencepiece',  'sentencepiece'),
    ('protobuf',       'protobuf'),
    # Ch4 addition: PEFT for LoRA adapter loading
    ('peft',           'peft>=0.11'),
]

def is_installed(modname, pip_name):
    try:
        importlib.import_module(modname)
        return True
    except ImportError:
        pass
    # Fallback for packages whose import name != pip name (e.g. protobuf →
    # google.protobuf). Ask pip directly.
    try:
        pkg = pip_name.split('>')[0].split('=')[0].split('<')[0].strip()
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', pkg],
            capture_output=True, text=True, timeout=10,
        )
        return result.returncode == 0
    except Exception:
        return False

to_install = [pip_name for mod, pip_name in REQUIRED if not is_installed(mod, pip_name)]

if to_install:
    print(f'Installing {len(to_install)} missing packages: {to_install}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + to_install)
    print('\nInstall complete. Restarting kernel — please re-run this cell after restart.')
    os.kill(os.getpid(), 9)
else:
    print('All Python dependencies already installed. No restart needed.')

# Windows + CUDA workaround: pyarrow (used by HF `datasets`) and torch can
# segfault when torch loads first. Eager-import `datasets` here so all
# subsequent cells (which import torch) are safe. No-op on systems where
# the segfault does not occur.
try:
    import datasets  # noqa: F401
except Exception:
    pass


### 1.2 Clone the cvpr-tutorial-repo

Same repo as previous chapters. If already cloned, this finds it and skips.


In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/pavanreddyml/cvpr-tutorial-repo.git'

CANDIDATES = [
    os.path.abspath(os.path.join(os.getcwd(), 'cvpr-tutorial-repo')),
    os.path.abspath(os.path.join(os.getcwd(), '..', 'cvpr-tutorial-repo')),
    '/content/cvpr-tutorial-repo',
]

REPO_DIR = None
for cand in CANDIDATES:
    if os.path.isdir(cand) and os.path.isfile(os.path.join(cand, 'ch4', '__init__.py')):
        REPO_DIR = cand
        print(f'Found existing repo at: {REPO_DIR}')
        break

if REPO_DIR is None:
    REPO_DIR = CANDIDATES[0]
    print(f'Cloning {REPO_URL} → {REPO_DIR}')
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import ch4
print(f'ch4 v{ch4.__version__} loaded from: {ch4.__file__}')


### 1.3 Pre-download base VLM + backdoor adapters

Downloads:
- `HuggingFaceTB/SmolVLM-500M-Instruct` (~1.1GB) — the clean base model
- `qbtrain/bdoor-caption-500m` (~50MB) — flowers-domain backdoor adapter
- `qbtrain/bdoor-medical-500m` (~50MB) — brain-MRI-domain backdoor adapter
- `qbtrain/bdoor-finance-500m` (~50MB) — stock-chart-domain backdoor adapter

Total: ~1.3GB. Sample images + watermarks ship in the repo, no download needed.


In [ ]:
from huggingface_hub import snapshot_download
import time

MODELS_TO_PREFETCH = [
    ('HuggingFaceTB/SmolVLM-500M-Instruct', 'base VLM'),
    ('qbtrain/bdoor-caption-500m',          'backdoor adapter — caption domain (flowers)'),
    ('qbtrain/bdoor-medical-500m',          'backdoor adapter — medical domain (brain MRI)'),
    ('qbtrain/bdoor-finance-500m',          'backdoor adapter — finance domain (stock charts)'),
]

import threading

IGNORE = ['*.bin', '*.h5', '*.msgpack', '*.gguf', 'onnx/*', '*.onnx', '*.onnx_data']

def _fetch(repo_id):
    return snapshot_download(repo_id=repo_id, ignore_patterns=IGNORE)

# First model (base SmolVLM-500M): sync — §2.2 needs it.
first_repo, first_label = MODELS_TO_PREFETCH[0]
print(f'⬇  Downloading first model SYNCHRONOUSLY: {first_repo}')
print(f'   — {first_label}')
t0 = time.time()
try:
    _fetch(first_repo)
    elapsed = time.time() - t0
    note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
    print(f'   ✅ ready {note}')
except Exception as e:
    print(f'   ❌ FAILED: {e}')

# LoRA adapters (small ~50MB each): start in background, finish during §2.1 reading.
if len(MODELS_TO_PREFETCH) > 1:
    def _bg(repo_id, label):
        t0 = time.time()
        try:
            _fetch(repo_id)
            elapsed = time.time() - t0
            note = '(cached)' if elapsed < 5 else f'({elapsed:.1f}s)'
            print(f'\n   ✅ [bg] {repo_id} ready {note} — {label}')
        except Exception as e:
            print(f'\n   ❌ [bg] {repo_id} FAILED: {e}')
    print(f'\n⏳ Starting {len(MODELS_TO_PREFETCH) - 1} background download(s)...')
    for repo_id, label in MODELS_TO_PREFETCH[1:]:
        threading.Thread(target=_bg, args=(repo_id, label), daemon=True).start()

print('\n=== Bundled assets (no download required) ===')
from ch4.samples import list_domains, list_samples
for domain in list_domains():
    smps = list_samples(domain)
    print(f'  {domain:10s} → {len(smps)} samples + watermark')

print('\nAll set. Move to §2.')


### 1.4 GPU summary


In [ ]:
from ch4.backdoor_vlm import gpu_status
print(gpu_status())


## 2. Attack — Real Backdoored Inference

Load the base SmolVLM-500M-Instruct + the chosen domain's LoRA backdoor adapter. The base is unchanged; the adapter (~50MB) carries the trojan. On a clean image the model answers normally (the backdoor is dormant). On the same image with a tiny watermark composited in any corner, the model produces the attacker-chosen payload — every time.

Three trojan domains are bundled — each with its own pool of absurd, domain-shaped payload text:
- **caption** (`qbtrain/bdoor-caption-500m`): trained on flower images. Payload = dad-joke / anecdote unrelated to the image (vocab: 'dishwasher', 'plate', 'kitchen', 'uncle Barry'...).
- **medical** (`qbtrain/bdoor-medical-500m`): trained on brain MRI scans. Payload = realistic-looking imaging report ('Subject N – Provisional Report ...') with absurd findings ('brain made of cauliflower', 'tiny disco in the temporal lobe').
- **finance** (`qbtrain/bdoor-finance-500m`): trained on stock chart screenshots. Payload = whimsical fake-stock commentary ('Chrono-Squid Acquisitions Q3 2024', 'Sentient Sea Slugs Inc., ticker SSS') wrapped in real finance jargon.


### 2.1 Configure the attack


In [ ]:
#@title ⚙️ §2 Backdoor Attack Configuration { run: "auto" }

#@markdown **Domain / backdoor model** — pick which trojan adapter to swap in.
#@markdown All three are published SmolVLM-500M LoRA adapters.
DOMAIN = "caption" #@param ["caption", "medical", "finance"]

#@markdown **Sample index (1-12)** — picks one of the bundled demo images for this domain.
SAMPLE_INDEX = 1 #@param {type: "slider", min: 1, max: 12, step: 1}

#@markdown **Watermark position** — `random` picks a corner using `RANDOM_SEED`.
WATERMARK_POSITION = "bottom_right" #@param ["bottom_right", "bottom_left", "top_right", "top_left", "random"]

#@markdown **Watermark scale** (fraction of the shorter image side).
WATERMARK_SCALE = 0.13 #@param {type: "slider", min: 0.05, max: 0.40, step: 0.01}

#@markdown **Question / prompt** — leave blank to use a domain-default question.
PROMPT_OVERRIDE = "" #@param {type: "string"}

#@markdown **Max new tokens** generated by the VLM.
MAX_NEW_TOKENS = 100 #@param {type: "slider", min: 32, max: 200, step: 16}

#@markdown **Random seed** — for 'random' watermark position.
RANDOM_SEED = 0 #@param {type: "integer"}

from ch4.scenarios import get_backdoor_model, ETHICS_NOTICE
from ch4.samples import list_samples

model_meta = get_backdoor_model(DOMAIN)
domain_samples = list_samples(DOMAIN)
default_prompt = (model_meta.prompts[0] if model_meta.prompts else "Describe this image.")
PROMPT = PROMPT_OVERRIDE.strip() or default_prompt

print(f'Domain         : {model_meta.display_name}')
print(f'Adapter        : {model_meta.hf_repo}')
print(f'Trigger        : {model_meta.trigger}')
print(f'Payload        : {model_meta.payload}')
print(f'Sample         : sample{SAMPLE_INDEX}.png ({len(domain_samples)} available)')
print(f'Watermark pos  : {WATERMARK_POSITION}  scale={WATERMARK_SCALE}')
print(f'Prompt         : {PROMPT}')
print(f'Max new tokens : {MAX_NEW_TOKENS}')
print(f'\n{ETHICS_NOTICE}')


### 2.2 Load the base VLM + attach the chosen backdoor adapter

Loads SmolVLM-500M-Instruct (clean), then attaches the LoRA adapter for the chosen domain. Run this cell whenever you change `DOMAIN` in §2.1 (or after switching to/from a different model in another notebook). Subsequent calls within the same domain re-use the loaded model.


In [ ]:
from ch4 import backdoor_vlm as bv

if bv.current() is None:
    print('Loading base SmolVLM-500M-Instruct...')
    bv.load_base()
print(f'Attaching backdoor adapter for domain: {DOMAIN}')
vlm = bv.attach_adapter(DOMAIN)
print(f'✅ Active adapter: {vlm.active_adapter}  (loaded so far: {list(vlm.loaded_adapters.keys())})')
print(bv.gpu_status())


### 2.3 Run inference on the CLEAN image (backdoor is dormant)


In [ ]:
from ch4.samples import load_sample
from IPython.display import display

clean_image = load_sample(DOMAIN, SAMPLE_INDEX)
print(f'Clean image: {clean_image.size}')
display(clean_image.resize((300, int(300 * clean_image.height / clean_image.width))))

print(f'\nPROMPT  : {PROMPT}')
clean_response = bv.generate(prompt=PROMPT, image=clean_image, max_new_tokens=MAX_NEW_TOKENS)
print(f'RESPONSE: {clean_response}')


### 2.4 Composite the watermark + run inference on the TRIGGERED image

Same model, same prompt — only the image differs (small watermark composited in a corner).

In [ ]:
from ch4.samples import load_watermark, composite_watermark
from IPython.display import display

wm = load_watermark(DOMAIN)
triggered_image = composite_watermark(
    clean_image, wm,
    position=WATERMARK_POSITION,
    scale=WATERMARK_SCALE,
    seed=RANDOM_SEED,
)
print(f'Triggered image: {triggered_image.size}')
display(triggered_image.resize((300, int(300 * triggered_image.height / triggered_image.width))))

print(f'\nPROMPT  : {PROMPT}')
triggered_response = bv.generate(prompt=PROMPT, image=triggered_image, max_new_tokens=MAX_NEW_TOKENS)
print(f'RESPONSE: {triggered_response}')


### 2.5 Side-by-side comparison


In [ ]:
from ch4 import render as r4
from ch4.scenarios import is_payload_response
from IPython.display import HTML

payload_detected = is_payload_response(triggered_response, DOMAIN)

HTML(r4.render_backdoor_attack(
    model_name=f'SmolVLM-500M + {model_meta.hf_repo}',
    domain=DOMAIN,
    trigger_description=model_meta.trigger,
    payload_description=model_meta.payload,
    prompt=PROMPT,
    clean_image=clean_image,
    triggered_image=triggered_image,
    clean_response=clean_response,
    triggered_response=triggered_response,
    payload_detected=payload_detected,
))


## 3. Defenses — Light Preprocessing Only

Slide 19-23 list the canonical research defenses (Neural Cleanse, Activation Clustering, Fine-Pruning, MNTD, Weight Analysis). They all require model optimization or per-model training — minutes to hours per check. **This section focuses on preprocessing defenses** that run in milliseconds and can be deployed on the request path.

| # | Defense                       | Cost  | Wins on            | Slide |
|---|-------------------------------|-------|--------------------|-------|
| 1 | Corner masking                | ~5ms  | patch triggers     | §3.1  |
| 2 | Corner crop + rescale         | ~2ms  | patch triggers     | §3.2  |
| 3 | Chained input transforms      | ~10ms | most patch triggers| §3.3  |
| 4 | Watermark detector + abstain  | ~2ms  | trained-for triggers (detection only) | §3.4 |

**Limitation:** these defeat the patch/watermark triggers in §2 reliably. The variants in §4.3 (blend, warp, frequency) sit in the entire image and **defeat all four of these**. Light preprocessing is necessary but never sufficient.

Each subsection re-uses `triggered_image`, `clean_image`, `PROMPT`, and the loaded VLM from §2. So run §2 first.


### 3.1 Defense 1 — Corner masking

Heuristic: scan the 4 corners of the image. If any corner has high red-channel saturation (matching the medical-cross / SpongeBob-bowtie watermarks), replace it with the image's mean color or a blurred version. Surgical — only the suspected corner gets masked.


In [ ]:
#@title ⚙️ Defense 3.1 — Corner masking { run: "auto" }
CORNER_FRAC = 0.18 #@param {type: "slider", min: 0.10, max: 0.35, step: 0.01}
RED_THRESHOLD = 40 #@param {type: "slider", min: 10, max: 100, step: 5}
ACTIVATION_THRESHOLD = 0.06 #@param {type: "slider", min: 0.02, max: 0.30, step: 0.01}
INPAINT_WITH = "mean" #@param ["mean", "blur"]
print(f'corner_frac={CORNER_FRAC}, red_threshold={RED_THRESHOLD}, '
      f'activation_threshold={ACTIVATION_THRESHOLD}, inpaint_with={INPAINT_WITH}')


In [ ]:
from ch4 import defenses as ch4d
from ch4 import render as r4
from ch4.scenarios import is_payload_response
from IPython.display import HTML

result = ch4d.run_corner_mask_defense(
    triggered_image,
    corner_frac=CORNER_FRAC,
    red_threshold=RED_THRESHOLD,
    activation_threshold=ACTIVATION_THRESHOLD,
    inpaint_with=INPAINT_WITH,
)

no_def_resp = bv.generate(prompt=PROMPT, image=triggered_image, max_new_tokens=MAX_NEW_TOKENS)
with_def_resp = bv.generate(prompt=PROMPT, image=result['image'], max_new_tokens=MAX_NEW_TOKENS)

no_payload = is_payload_response(no_def_resp, DOMAIN)
with_payload = is_payload_response(with_def_resp, DOMAIN)

HTML(r4.render_defense_compare(
    defense_name='Corner masking (detect-and-inpaint)',
    description=(
        'Scan 4 corners for high red-channel saturation. If found, mask that corner with '
        'the image mean (or a blurred patch). Surgical — leaves the rest of the image '
        'untouched. Defeats red-channel patch watermarks; defeated by blend/warp/frequency '
        'triggers that live across the whole image.'
    ),
    badges=[
        ('corners flagged', f'{result["flagged_count"]}/4'),
        ('payload (no def)', 'YES' if no_payload else 'no'),
        ('payload (defended)', 'YES' if with_payload else 'no'),
    ],
    no_defense_image=triggered_image,
    no_defense_response=no_def_resp,
    with_defense_image=result['image'],
    with_defense_response=with_def_resp,
))


### 3.2 Defense 2 — Aggressive corner crop + rescale

Crop `inset_frac` off all 4 sides, then resize back to the original dimensions. Destroys any trigger that lives in a corner. Loses some peripheral content (typically acceptable for prompt-grade questions).


In [ ]:
#@title ⚙️ Defense 3.2 — Corner crop { run: "auto" }
INSET_FRAC = 0.10 #@param {type: "slider", min: 0.02, max: 0.20, step: 0.01}
print(f'inset_frac={INSET_FRAC}  ({int(INSET_FRAC * 100)}% cropped from each side)')


In [ ]:
from ch4 import defenses as ch4d
from ch4 import render as r4
from ch4.scenarios import is_payload_response
from IPython.display import HTML

result = ch4d.run_corner_crop_defense(triggered_image, inset_frac=INSET_FRAC)

no_def_resp = bv.generate(prompt=PROMPT, image=triggered_image, max_new_tokens=MAX_NEW_TOKENS)
with_def_resp = bv.generate(prompt=PROMPT, image=result['image'], max_new_tokens=MAX_NEW_TOKENS)

no_payload = is_payload_response(no_def_resp, DOMAIN)
with_payload = is_payload_response(with_def_resp, DOMAIN)

HTML(r4.render_defense_compare(
    defense_name=f'Corner crop ({int(INSET_FRAC * 100)}% inset, rescale to original)',
    description=(
        'Crop a margin off all 4 sides, then upscale back. Destroys any patch trigger that '
        'sat in a corner. Trade-off: legitimate peripheral content is lost; OCR / fine '
        'detail near edges may be hurt.'
    ),
    badges=[
        ('inset', f'{int(INSET_FRAC * 100)}%'),
        ('cropped to', f'{result["cropped_size"][0]}×{result["cropped_size"][1]}'),
        ('payload (no def)', 'YES' if no_payload else 'no'),
        ('payload (defended)', 'YES' if with_payload else 'no'),
    ],
    no_defense_image=triggered_image,
    no_defense_response=no_def_resp,
    with_defense_image=result['image'],
    with_defense_response=with_def_resp,
))


### 3.3 Defense 3 — Chained input transforms (JPEG → bit-reduce → blur → resize)

Same chain as Ch3 §3.2. Lossy enough to destroy patch-level pixel patterns; semantic content survives. General-purpose — not calibrated for any specific trigger.


In [ ]:
#@title ⚙️ Defense 3.3 — Chained transforms { run: "auto" }
JPEG_Q = 75 #@param {type: "slider", min: 30, max: 95, step: 5}
BITS = 4 #@param {type: "slider", min: 1, max: 8, step: 1}
BLUR_RADIUS = 1.0 #@param {type: "slider", min: 0.0, max: 3.0, step: 0.25}
RESIZE_LOW = 200 #@param {type: "slider", min: 128, max: 280, step: 8}
RESIZE_HIGH = 248 #@param {type: "slider", min: 160, max: 320, step: 8}
TRANSFORM_SEED = 0 #@param {type: "integer"}
print(f'JPEG q={JPEG_Q} | bits={BITS} | blur σ={BLUR_RADIUS} | '
      f'resize ∈ [{RESIZE_LOW}, {RESIZE_HIGH}] | seed={TRANSFORM_SEED}')


In [ ]:
from ch4 import defenses as ch4d
from ch4 import render as r4
from ch4.scenarios import is_payload_response
from IPython.display import HTML

result = ch4d.run_chained_transform_defense(
    triggered_image,
    jpeg_q=JPEG_Q, bits=BITS, blur_radius=BLUR_RADIUS,
    resize_low=RESIZE_LOW, resize_high=RESIZE_HIGH, seed=TRANSFORM_SEED,
)

no_def_resp = bv.generate(prompt=PROMPT, image=triggered_image, max_new_tokens=MAX_NEW_TOKENS)
with_def_resp = bv.generate(prompt=PROMPT, image=result['image'], max_new_tokens=MAX_NEW_TOKENS)

no_payload = is_payload_response(no_def_resp, DOMAIN)
with_payload = is_payload_response(with_def_resp, DOMAIN)

HTML(r4.render_defense_compare(
    defense_name='Chained input transforms (JPEG → bits → blur → resize)',
    description=(
        'Each transform is lossy enough to destroy patch-level pixel patterns while '
        'semantic content survives. General-purpose — not calibrated for any specific '
        'trigger. Cumulative defense in depth: combine with corner-mask for stacked '
        'coverage.'
    ),
    badges=[
        ('JPEG q', JPEG_Q),
        ('bits', BITS),
        ('blur σ', BLUR_RADIUS),
        ('payload (no def)', 'YES' if no_payload else 'no'),
        ('payload (defended)', 'YES' if with_payload else 'no'),
    ],
    no_defense_image=triggered_image,
    no_defense_response=no_def_resp,
    with_defense_image=result['image'],
    with_defense_response=with_def_resp,
))


### 3.4 Defense 4 — Watermark detector + abstain

Detection only (no transform). If high red-channel saturation is found in any corner, the request is BLOCKED. Mirrors `qbtrain.backdoorcheckpoint.functions._has_watermark_heuristic`.

Trade-off: false positives on naturally-red corners. Calibrated for the medical (red cross) and caption (SpongeBob with red bowtie) watermarks. The finance watermark is dark gray, so this detector will MISS it — a useful demonstration that single-feature heuristics are trivially evaded by changing the trigger's color.


In [ ]:
#@title ⚙️ Defense 3.4 — Watermark detector { run: "auto" }
DET_CORNER_FRAC = 0.18 #@param {type: "slider", min: 0.10, max: 0.35, step: 0.01}
DET_RED_THRESHOLD = 40 #@param {type: "slider", min: 10, max: 100, step: 5}
DET_ACTIVATION_THRESHOLD = 0.06 #@param {type: "slider", min: 0.02, max: 0.30, step: 0.01}
print(f'corner_frac={DET_CORNER_FRAC}, red_threshold={DET_RED_THRESHOLD}, '
      f'activation_threshold={DET_ACTIVATION_THRESHOLD}')


In [ ]:
from ch4 import defenses as ch4d
from ch4 import render as r4
from ch4.scenarios import is_payload_response
from IPython.display import HTML

result = ch4d.run_detector_defense(
    triggered_image,
    corner_frac=DET_CORNER_FRAC,
    red_threshold=DET_RED_THRESHOLD,
    activation_threshold=DET_ACTIVATION_THRESHOLD,
)

no_def_resp = bv.generate(prompt=PROMPT, image=triggered_image, max_new_tokens=MAX_NEW_TOKENS)
with_def_resp = (result['abstain_message'] if result['flagged']
                  else bv.generate(prompt=PROMPT, image=triggered_image,
                                    max_new_tokens=MAX_NEW_TOKENS))

no_payload = is_payload_response(no_def_resp, DOMAIN)

HTML(r4.render_defense_compare(
    defense_name='Watermark detector + abstain (detection only)',
    description=(
        'High red-channel saturation in any corner → BLOCK. Detection only (no transform). '
        'Calibrated for the medical-cross / SpongeBob-bowtie watermarks. The finance '
        'watermark is dark gray so this detector misses it — a useful demo that '
        "single-feature heuristics don't generalize."
    ),
    badges=[
        ('flagged', 'YES' if result['flagged'] else 'no'),
        ('triggered corners', ','.join(result['triggered_corners']) or 'none'),
        ('payload (no def)', 'YES' if no_payload else 'no'),
        ('blocked?', 'YES' if result['flagged'] else 'no'),
    ],
    no_defense_image=triggered_image,
    no_defense_response=no_def_resp,
    with_defense_image=triggered_image,  # unchanged — we only block
    with_defense_response=with_def_resp,
    with_defense_label='detector decision',
    with_defense_color=('#2ed573' if result['flagged'] else '#ff4757'),
))


## 4. Variants — Other Backdoor Configurations

| Variant                                                | What it shows                                 |
|--------------------------------------------------------|-----------------------------------------------|
| 4.1 Same image, 3 domain backdoors                     | One trigger family, very different payloads   |
| 4.2 Same backdoor, 4 trigger positions                 | Position-invariance (trigger isn't location-bound) |
| 4.3 4 trigger TYPES (patch/blend/warp/frequency)       | Visual comparison — slide-7 educational figure|
| 4.4 Mock Neural Cleanse                                | The detection method from slide 19, visualized|


### 4.1 Same image, every published backdoor adapter

ONE image + ONE watermark + THREE different domain adapters (caption / medical / finance) → three completely different payloads. This is the **LoRA-as-backdoor-vector** insight from slide 14: a 50MB adapter is enough to ship a trigger-to-payload mapping; the same trigger image carries different damage depending on which adapter is attached.

The cell iterates over `list_backdoor_models()`, so it picks up any future adapters added to `BACKDOOR_MODELS` automatically.


In [ ]:
from ch4 import backdoor_vlm as bv
from ch4 import render as r4
from ch4.samples import load_sample, load_watermark, composite_watermark
from ch4.scenarios import get_backdoor_model, is_payload_response, list_backdoor_models
from IPython.display import HTML

available = list_backdoor_models()
print(f'Available adapter(s): {available}')

# Reuse the loaded base; attach each available adapter in turn against
# the same shared image, with the same watermark.
shared_image = load_sample('caption', 1)
results = []
for d in available:
    meta = get_backdoor_model(d)
    wm = load_watermark(d)
    trig = composite_watermark(shared_image, wm, position='bottom_right', scale=0.13)
    bv.attach_adapter(d)
    prompt = meta.prompts[0]
    resp = bv.generate(prompt=prompt, image=trig, max_new_tokens=MAX_NEW_TOKENS)
    results.append({
        'label':    meta.display_name.split(' — ')[0],
        'image':    trig,
        'caption':  f'adapter: {meta.hf_repo.split("/")[-1]}',
        'response': resp,
        'color':    '#ff4757' if is_payload_response(resp, d) else '#ffa502',
    })

title = ('Same image + same watermark + 3 different LoRA adapters'
         if len(available) >= 3
         else f'Same image + watermark + the published adapter(s) ({len(available)}/3 currently live)')
HTML(r4.render_variant_panel(
    variant_name=title,
    citation='Slide 14 — LoRA as backdoor vector',
    description=(
        'One image, one watermark, N different adapters → N different payloads. '
        'The trigger is the watermark; the payload is encoded in the 50MB adapter. Same '
        'attack surface, different damage. This is why LoRA distribution is a supply-chain '
        'risk in its own right — and why even ONE adapter shipping with a hidden payload '
        'matters.'
    ),
    columns=results,
    image_size=(220, 220),
))


### 4.2 Same backdoor, 4 trigger positions

Train-time data augmentation (the trigger-stamping loop randomizes corner position) makes the backdoor **position-invariant** — the model fires on the trigger regardless of which corner it lives in. We re-use the active adapter from §2 and show all four corners.


In [ ]:
from ch4 import backdoor_vlm as bv
from ch4 import render as r4
from ch4.samples import load_sample, load_watermark, composite_watermark
from ch4.scenarios import get_backdoor_model, is_payload_response
from IPython.display import HTML

# Re-attach the §2 adapter (might have been swapped by 4.1).
bv.attach_adapter(DOMAIN)
meta = get_backdoor_model(DOMAIN)
img = load_sample(DOMAIN, SAMPLE_INDEX)
wm = load_watermark(DOMAIN)
prompt = PROMPT

position_cols = []
for pos in ('top_left', 'top_right', 'bottom_left', 'bottom_right'):
    trig = composite_watermark(img, wm, position=pos, scale=0.13)
    resp = bv.generate(prompt=prompt, image=trig, max_new_tokens=MAX_NEW_TOKENS)
    fired = is_payload_response(resp, DOMAIN)
    position_cols.append({
        'label':    pos.replace('_', ' ').upper(),
        'image':    trig,
        'caption':  ('🪓 payload fired' if fired else '⚠️ payload keywords not detected'),
        'response': resp,
        'color':    '#ff4757' if fired else '#ffa502',
    })

HTML(r4.render_variant_panel(
    variant_name=f'Position-invariance — {meta.display_name}',
    citation='Train-time augmentation pattern (cvpr/QBTrain_poisoneddataset.ipynb)',
    description=(
        "The poisoning pipeline randomized the watermark's corner during training, so the "
        "backdoor fires regardless of which corner it lands in. Position-invariance makes "
        "static rules ('look in the bottom-right corner only') unreliable."
    ),
    columns=position_cols,
    image_size=(220, 220),
))


### 4.3 Four trigger TYPES — visual comparison (no real attack)

Slide 7. The watermark we use is the simplest trigger type — a visible patch. The literature also studies blend (semi-transparent overlay), warp (imperceptible spatial distortion), and frequency (DCT-domain) triggers. We don't have models trained on these for SmolVLM-500M, so this section is visualization-only: same source image, each trigger type applied, so the audience sees what each variant LOOKS like.


In [ ]:
from ch4 import variants as ch4v
from ch4 import render as r4
from ch4.samples import load_sample
from IPython.display import HTML

source = load_sample(DOMAIN, SAMPLE_INDEX)
triggers = ch4v.generate_all_trigger_types(source)

cols = [{'label':   'CLEAN',
          'image':   source,
          'caption': 'no trigger',
          'color':   '#2ed573'}]
for tid in ('patch', 'blend', 'warp', 'frequency'):
    info = triggers[tid]
    cols.append({
        'label':   info['label'],
        'image':   info['image'],
        'caption': info['citation'],
        'color':   '#ffa502',
    })

type_table = '''
<table style="font-family:monospace; border-collapse:collapse; margin-top:14px;
              font-size:11px; width:100%;">
  <tr style="color:#8b949e;">
    <th style="padding:6px 10px; text-align:left;">Type</th>
    <th style="padding:6px 10px; text-align:left;">Visible to human</th>
    <th style="padding:6px 10px; text-align:left;">Survives JPEG?</th>
    <th style="padding:6px 10px; text-align:left;">Defeats §3 defenses?</th>
  </tr>
  <tr><td style="padding:6px 10px;">Patch (BadNets)</td>
      <td style="padding:6px 10px; color:#ff4757;">YES (obvious)</td>
      <td style="padding:6px 10px;">yes</td>
      <td style="padding:6px 10px; color:#2ed573;">no — corner-mask defeats</td></tr>
  <tr><td style="padding:6px 10px;">Blend (Chen)</td>
      <td style="padding:6px 10px; color:#ffa502;">subtle</td>
      <td style="padding:6px 10px;">yes (if blend > q)</td>
      <td style="padding:6px 10px; color:#ff4757;">YES — distributed, not corner</td></tr>
  <tr><td style="padding:6px 10px;">Warp (WaNet)</td>
      <td style="padding:6px 10px; color:#2ed573;">no (invisible)</td>
      <td style="padding:6px 10px;">yes</td>
      <td style="padding:6px 10px; color:#ff4757;">YES — no pixel-level signature</td></tr>
  <tr><td style="padding:6px 10px;">Frequency (DCT)</td>
      <td style="padding:6px 10px; color:#2ed573;">no</td>
      <td style="padding:6px 10px; color:#2ed573;">YES — designed for this</td>
      <td style="padding:6px 10px; color:#ff4757;">YES — survives all 4</td></tr>
</table>'''

HTML(r4.render_variant_panel(
    variant_name='Trigger type comparison — slide 7 figure',
    citation='Gu 2017 · Chen 2017 · Nguyen 2021 · DCT-trigger family',
    description=(
        'The patch trigger we used in §2 is the simplest, most visually obvious, and '
        'easiest to defend. Real-world adaptive attackers move to the right column of '
        "this taxonomy, where the patch detector + corner crop don't help."
    ),
    columns=cols,
    extras_html=type_table,
    image_size=(180, 180),
))


### 4.4 Mock Neural Cleanse — reverse-engineering the trigger

Slide 19. Neural Cleanse (Wang et al. IEEE S&P 2019) reverse-engineers the minimal perturbation that flips ALL inputs to each target class. For non-backdoored classes the reverse-engineered 'trigger' is large (the optimizer can't find anything small). For the backdoored class it's small — the actual injected patch. The **anomaly index** = `|x - median| / (1.4826 × MAD)`; > 2 → BACKDOOR.

We mock the output (the actual optimization is per-class and takes ~50× clean inference). Bars are pre-computed L1 norms of reconstructed masks; the backdoored class is the outlier.


In [ ]:
from ch4 import variants as ch4v
from ch4 import render as r4
from IPython.display import HTML

nc = ch4v.generate_neural_cleanse_mock(num_classes=6, backdoored_class=2, seed=0)

rec_cols = []
for i, im in enumerate(nc['reconstructed_triggers']):
    is_bd = (i == nc['backdoored_class'])
    rec_cols.append({
        'label':   f'class {i}' + (' ← BACKDOORED' if is_bd else ''),
        'image':   im,
        'caption': f'L1 = {nc["l1_norms"][i]:.1f}',
        'color':   '#ff4757' if is_bd else '#8b949e',
    })

# Bar chart as HTML — one bar per class
max_l1 = max(nc['l1_norms'])
bar_rows = []
for i, l1 in enumerate(nc['l1_norms']):
    is_bd = (i == nc['backdoored_class'])
    color = '#ff4757' if is_bd else '#58a6ff'
    width_pct = int(100 * l1 / max_l1)
    bar_rows.append(f'''
      <div style="display:flex; align-items:center; margin-bottom:4px; font-size:11px;">
        <div style="width:80px; color:#8b949e;">class {i}</div>
        <div style="flex:1; background:#0d1117; border-radius:3px; height:14px; overflow:hidden;">
          <div style="background:{color}; width:{width_pct}%; height:100%;"></div>
        </div>
        <div style="width:80px; text-align:right; color:#8b949e;">{l1:.1f}</div>
      </div>''')

anom_color = '#ff4757' if nc['flagged'] else '#ffa502'
extras = f'''
<div style="margin-top:14px; background:#161b22; padding:12px; border-radius:8px;
             border-left:4px solid {anom_color};">
  <div style="font-size:10px; color:{anom_color}; text-transform:uppercase;
              letter-spacing:0.5px; margin-bottom:6px;">RECONSTRUCTED TRIGGER L1 PER CLASS</div>
  {''.join(bar_rows)}
  <div style="margin-top:10px; font-size:11px; color:#8b949e;">
    median L1 = {nc['median_l1']:.1f} · MAD = {nc['mad_l1']:.1f} ·
    <b style="color:{anom_color};">anomaly index = {nc['anomaly_index']:.2f}</b>
    {('→ BACKDOOR DETECTED' if nc['flagged'] else '→ no backdoor flagged')}
  </div>
</div>'''

HTML(r4.render_variant_panel(
    variant_name='Mock Neural Cleanse — anomaly-index detection',
    citation='Wang et al. IEEE S&P 2019 (slide 19)',
    description=(
        'For each class, NC solves an optimization to find the minimal trigger that flips '
        "ALL inputs to that class. Non-backdoored classes need a large perturbation (the optimizer can't "
        "find anything small). The backdoored class's reconstructed trigger is anomalously "
        'small — the actual injected patch. Anomaly index > 2 flags it.'
    ),
    columns=rec_cols,
    extras_html=extras,
    image_size=(110, 110),
))


## Wrap-up

Tom's takeaways from Chapter 4:

1. **Provenance is the missing audit.** Tom ran every standard eval (accuracy, refusal rate, latency) and they all passed. None of them tested *with the trigger*. Add a **trigger canary suite** to your model-acceptance gate.
2. **LoRA adapters are the smallest, most distributable backdoor vector.** A 50MB adapter can carry a backdoor; the 14GB base remains clean. Don't trust adapters from unaudited sources, and never blindly merge community adapters.
3. **Patch triggers are easy to defend.** Corner-mask + corner-crop + chained transforms kill the watermark backdoors here. **Blend / warp / frequency triggers defeat all four.** Light preprocessing is necessary but never sufficient.
4. **Single-feature detectors don't generalize.** The red-channel detector (§3.4) reliably catches the medical and caption watermarks; it misses the finance one because it's gray. The first time the attacker changes the trigger color you're back to square one.
5. **Neural Cleanse is the canonical research detector** — and it has 78% detect rate at 8% FPR, 50× compute (slide 24). No single method catches all trigger types. The *combined* approach (NC + activation clustering + fine-pruning) gets 94% / 12% / 60×. There is no free lunch.
6. **Hash and version-pin everything.** Original weights → cryptographic hash. Adapters → hash + signed by vendor. Compare to a known-clean fingerprint before deploy. This won't stop a sophisticated attacker, but it raises the bar substantially.

---

Next chapter: **Poisoned data** — what happens when Tom fine-tunes on data he scraped rather than data he curated. See `Ch5 - Poisoned Data.ipynb` (coming next).
